### Example 8 for the usage of Rela²x.
### See README.md for more information.

- $^{1}\text{H}$ and $^{13}\text{C}$ nuclei.

- Dipole-dipole coupling and $^{13}\text{C}$ chemical-shift anisotropy under isotropic rotational diffusion.

- Relaxation rates in terms of the interaction parameters, with `substitute_G0`.

Import Rela²x, and *SymPy* for the last step.

In [1]:
from rela2x import *
import sympy as smp

Define the spin system and the incoherent interactions. The names `'DD'` and `'CSA'` tell `substitute_G0` what the interactions are.

In [2]:
spin_system = ['1H', '13C']

incoh_intrs = {'DD':  ('2',  [[0, 1],
                              [0, 0]], [2]),
               'CSA': ('1L', [0, 1], [2])}

Compute $R$ and apply the isotropic rotational diffusion model.

In [3]:
R = relaxation_superoperator(spin_system, incoh_intrs)
R.to_isotropic_rotational_diffusion()

################################################################################
                          Relaxation superoperator (R)                          
################################################################################

Computing R for incoherent interaction pairs...
  DD12 * DD12
  DD12 * CSA2
  CSA2 * DD12
  CSA2 * CSA2
R computed in 0.19 seconds.

Changing basis...
Basis changed in 0.09 seconds.

Final clean-ups...
Done in 0.29 seconds.



The longitudinal relaxation rate of $^{13}\text{C}$ still contains the abstract amplitudes $G(0)$.

In [4]:
R.rate('210')

\tau_c*G^{DD12}_{20}(0)/(2*(\tau_c**2*(\omega_{13C} + \omega_{1H})**2 + 1)) + \tau_c*G^{DD12}_{20}(0)/(12*(\tau_c**2*(\omega_{13C} - \omega_{1H})**2 + 1)) + \tau_c*G^{CSA2}_{20}(0)/(\omega_{13C}**2*\tau_c**2 + 1) + \tau_c*G^{DD12}_{20}(0)/(4*(\omega_{13C}**2*\tau_c**2 + 1))

Substitute them. Every interaction is taken to be axially symmetric.

In [5]:
R.substitute_G0()

Substituting G(0)...
  CSA2: nuclear shielding
  DD12: dipole-dipole coupling



In [6]:
R.rate('210')

2*\Delta\sigma^{(2)}**2*\omega_{13C}**2*\tau_c/(15*(\omega_{13C}**2*\tau_c**2 + 1)) + 3*\tau_c*b_{12}**2/(5*(\tau_c**2*(\omega_{13C} + \omega_{1H})**2 + 1)) + \tau_c*b_{12}**2/(10*(\tau_c**2*(\omega_{13C} - \omega_{1H})**2 + 1)) + 3*\tau_c*b_{12}**2/(10*(\omega_{13C}**2*\tau_c**2 + 1))

The cross-relaxation rate between $^{1}\text{H}$ and $^{13}\text{C}$ is purely dipolar.

In [7]:
R.rate('110', '210')

3*\tau_c*b_{12}**2/(5*(\tau_c**2*(\omega_{13C} + \omega_{1H})**2 + 1)) - \tau_c*b_{12}**2/(10*(\tau_c**2*(\omega_{13C} - \omega_{1H})**2 + 1))

The cross-correlation of the two mechanisms connects $^{13}\text{C}$ magnetisation to the two-spin order, and depends on the angle $\theta$ between the symmetry axes of the two interactions.

In [8]:
R.rate('210', '110*210')

2*\Delta\sigma^{(2)}*\omega_{13C}*\tau_c*b_{12}*(3*cos(\theta_{DD12, CSA2})**2 - 1)/(5*(\omega_{13C}**2*\tau_c**2 + 1))

A shielding tensor without axial symmetry is given as a full coupling tensor. Here the dipolar axis is the $z$-axis of the shielding tensor, and $\eta$ is the asymmetry parameter.

In [9]:
eta = smp.Symbol('eta', real=True)
w_C = w_symbol('13C')
delta_sigma = Delta_sigma_symbol(2)

sigma = smp.Rational(2, 3)*delta_sigma*smp.diag(-(1 + eta)/2, -(1 - eta)/2, 1)
A_CSA = -w_C*sigma
A_DD = axial_tensor(-3*b_coupling_symbol(1, 2), [0, 0, 1])

In [10]:
R_asym = relaxation_superoperator(spin_system, incoh_intrs)
R_asym.to_isotropic_rotational_diffusion()
R_asym.substitute_G0({'CSA2': A_CSA, 'DD12': A_DD})

################################################################################
                          Relaxation superoperator (R)                          
################################################################################

Computing R for incoherent interaction pairs...
  DD12 * DD12
  DD12 * CSA2
  CSA2 * DD12
  CSA2 * CSA2
R computed in 0.20 seconds.

Changing basis...
Basis changed in 0.06 seconds.

Final clean-ups...
Done in 0.35 seconds.

Substituting G(0)...
  CSA2: coupling tensor given
  DD12: coupling tensor given



In [11]:
R_asym.rate('210')

2*\Delta\sigma^{(2)}**2*\omega_{13C}**2*\tau_c*(eta**2 + 3)/(45*(\omega_{13C}**2*\tau_c**2 + 1)) + 3*\tau_c*b_{12}**2/(5*(\tau_c**2*(\omega_{13C} + \omega_{1H})**2 + 1)) + \tau_c*b_{12}**2/(10*(\tau_c**2*(\omega_{13C} - \omega_{1H})**2 + 1)) + 3*\tau_c*b_{12}**2/(10*(\omega_{13C}**2*\tau_c**2 + 1))